# Experiment 1 — LR vs BERT on the 2,858 Idea-Mapped Samples

Benchmark of the 2,858 QSBC idea-mapped samples (single-label, 12 CPRA
classes, balanced 195–264/class). A **document-level 60/40** train/test
split (no leakage: all sentences of a policy doc stay on one side),
then two classifiers:

1. **TF-IDF + Logistic Regression** (sklearn) — fast baseline.
2. **Fine-tuned BERT** (`bert-base-uncased`) — HuggingFace Trainer,
   3 epochs, lr 2e-5.

Metrics on the held-out test set: **accuracy + macro-F1**.
Reference (full 37,284-corpus) targets: LR 0.7935 / 0.7161,
BERT 0.8208 / 0.7529. This experiment checks subset fidelity: does the
idea-mapped slice reproduce the full-corpus ranking?


In [1]:
# 1. Setup + fetch the committed sample set
import os
os.chdir('/content')
if not os.path.exists('/content/qsbc'):
    !git clone -q https://github.com/jpeckenpaugh/qsbc.git qsbc
os.chdir('/content/qsbc')
print('cwd:', os.getcwd())


cwd: /content/qsbc


In [2]:
!pip install -q pandas scikit-learn transformers datasets torch accelerate
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('results/exp1/samples_2858.csv')
df = df.drop_duplicates(subset=['sentence_id']).reset_index(drop=True)
print(df.shape)
print(df['label'].value_counts())


(2858, 4)
label
Categories of Personal Information Shared / Disclosed              264
Categories of Personal Information Collected                       264
Categories of Personal Information Sold                            251
Methods to exercise rights                                         250
Updated Privacy Policy                                             247
Description of Right to Delete                                     245
Description of Right to Know PI Collected                          244
Description of Right to Opt-out of sale of PI                      240
Description of Right to Non-discrimination on exercising rights    223
Description of Right to Correct Information                        222
Description of Right to Know PI sold / shared                      213
Description of Right to Limit use of PI                            195
Name: count, dtype: int64


## 2. Document-level 60/40 split

Split by `doc_key` (not by row) to prevent leakage between train and test.

In [3]:
from sklearn.model_selection import train_test_split

SEED = 42
docs = df['doc_key'].unique()
train_docs, test_docs = train_test_split(docs, test_size=0.4,
                                          random_state=SEED)
train_df = df[df['doc_key'].isin(train_docs)].reset_index(drop=True)
test_df  = df[df['doc_key'].isin(test_docs)].reset_index(drop=True)

# Leakage check: doc overlap must be 0
print('train docs:', len(train_docs), '| test docs:', len(test_docs))
print('doc overlap:', len(set(train_docs) & set(test_docs)))
print('train samples:', len(train_df), '| test samples:', len(test_df))
print('train classes:', train_df['label'].nunique(),
      '| test classes:', test_df['label'].nunique())
print('\nclass distribution (%):')
dist = pd.DataFrame({
    'train': train_df['label'].value_counts(normalize=True) * 100,
    'test': test_df['label'].value_counts(normalize=True) * 100,
}).fillna(0).round(2)
print(dist)


train docs: 234 | test docs: 156
doc overlap: 0
train samples: 1748 | test samples: 1110
train classes: 12 | test classes: 12

class distribution (%):
                                                    train   test
label                                                           
Categories of Personal Information Collected         8.87   9.82
Categories of Personal Information Shared / Dis...   9.32   9.10
Categories of Personal Information Sold              9.61   7.48
Description of Right to Correct Information          7.38   8.38
Description of Right to Delete                       8.35   8.92
Description of Right to Know PI Collected            8.87   8.02
Description of Right to Know PI sold / shared        8.58   5.68
Description of Right to Limit use of PI              7.38   5.95
Description of Right to Non-discrimination on e...   7.32   8.56
Description of Right to Opt-out of sale of PI        8.47   8.29
Methods to exercise rights                           8.07   9.82
Upda

In [4]:
y_train = train_df['label'].astype(str)
y_test  = test_df['label'].astype(str)
classes = sorted(set(y_train) | set(y_test))
print('num classes:', len(classes))


num classes: 12


## 3. Baseline — TF-IDF + Logistic Regression

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report

lr_pipe = make_pipeline(
    TfidfVectorizer(sublinear_tf=True, min_df=2, ngram_range=(1, 2),
                    stop_words='english'),
    LogisticRegression(max_iter=1000, C=10, class_weight='balanced'),
)
lr_pipe.fit(train_df['sentence'], y_train)
pred_lr = lr_pipe.predict(test_df['sentence'])
print('LR accuracy:', round(accuracy_score(y_test, pred_lr), 4))
print('LR macro-F1:', round(f1_score(y_test, pred_lr, average='macro'), 4))


LR accuracy: 0.6559
LR macro-F1: 0.6581


## 4. Fine-tuned BERT

`bert-base-uncased`, 3 epochs, lr 2e-5, batch 16. Uses the HF Trainer.

In [6]:
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments,
)

id2label = {i: c for i, c in enumerate(classes)}
label2id = {c: i for i, c in id2label.items()}

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

def tokenize(batch):
    return tokenizer(batch['sentence'], truncation=True, padding='max_length',
                     max_length=128)

train_ds = Dataset.from_pandas(
    pd.DataFrame({'sentence': train_df['sentence'],
                  'label': train_df['label'].map(label2id)}))
test_ds = Dataset.from_pandas(
    pd.DataFrame({'sentence': test_df['sentence'],
                  'label': test_df['label'].map(label2id)}))

train_ds = train_ds.map(tokenize, batched=True)
test_ds  = test_ds.map(tokenize, batched=True)
train_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

model = AutoModelForSequenceClassification.from_pretrained(
    'bert-base-uncased', num_labels=len(classes), id2label=id2label,
    label2id=label2id)

args = TrainingArguments(
    output_dir='/content/bert_out',
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    seed=SEED,
    report_to=[],
    save_strategy='no',
    logging_steps=50,
)
trainer = Trainer(model=model, args=args,
                  train_dataset=train_ds, eval_dataset=test_ds)
trainer.train()


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/1748 [00:00<?, ? examples/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
50,2.448646
100,2.157568
150,1.700065
200,1.437682
250,1.184926
300,1.111972


TrainOutput(global_step=330, training_loss=1.6182715965039802, metrics={'train_runtime': 28.4293, 'train_samples_per_second': 184.457, 'train_steps_per_second': 11.608, 'total_flos': 344969564221440.0, 'train_loss': 1.6182715965039802, 'epoch': 3.0})

In [7]:
import torch

out = trainer.predict(test_ds)
pred_bert = [id2label[int(x)] for x in np.argmax(out.predictions, axis=1)]
print('BERT accuracy:', round(accuracy_score(y_test, pred_bert), 4))
print('BERT macro-F1:', round(f1_score(y_test, pred_bert, average='macro'), 4))


BERT accuracy: 0.6523
BERT macro-F1: 0.643


In [8]:
# 5. Comparison table + per-class detail
rows = [
    ['TF-IDF + Logistic Regression',
     round(accuracy_score(y_test, pred_lr), 4),
     round(f1_score(y_test, pred_lr, average='macro'), 4)],
    ['Fine-tuned BERT',
     round(accuracy_score(y_test, pred_bert), 4),
     round(f1_score(y_test, pred_bert, average='macro'), 4)],
]
import pandas as pd
summary = pd.DataFrame(rows, columns=['Model', 'Accuracy', 'Macro-F1'])
print(summary.to_string(index=False))
print('\nReference (full 37,284 corpus): LR 0.7935 / 0.7161, BERT 0.8208 / 0.7529')
print('\nPer-class (LR):')
print(classification_report(y_test, pred_lr, zero_division=0))
print('\nPer-class (BERT):')
print(classification_report(y_test, pred_bert, zero_division=0))


                       Model  Accuracy  Macro-F1
TF-IDF + Logistic Regression    0.6559    0.6581
             Fine-tuned BERT    0.6523    0.6430

Reference (full 37,284 corpus): LR 0.7935 / 0.7161, BERT 0.8208 / 0.7529

Per-class (LR):
                                                                 precision    recall  f1-score   support

                   Categories of Personal Information Collected       0.47      0.57      0.51       109
          Categories of Personal Information Shared / Disclosed       0.44      0.50      0.47       101
                        Categories of Personal Information Sold       0.46      0.55      0.50        83
                    Description of Right to Correct Information       0.87      0.83      0.85        93
                                 Description of Right to Delete       0.93      0.79      0.85        99
                      Description of Right to Know PI Collected       0.59      0.45      0.51        89
                  Descript